In [1]:
!pip install mrjob

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 439.6/439.6 kB 7.1 MB/s eta 0:00:00


Q1. Word Count - Count the frequency of each word
Input:
• hadoop is fast
• hadoop is scalable

In [2]:
from collections import defaultdict

data = ["hadoop is fast", "hadoop is scalable"]

# 🔸 Mapper
mapped = []
for line in data:
    words = line.split()
    for word in words:
        mapped.append((word, 1))

print("Mapped:", mapped)

# 🔸 Shuffle & Sort
shuffled = defaultdict(list)
for key, value in mapped:
    shuffled[key].append(value)

print("Shuffled:", dict(shuffled))

# 🔸 Reducer
result = {}
for key, values in shuffled.items():
    result[key] = sum(values)

print("Final Output:", result)

Mapped: [('hadoop', 1), ('is', 1), ('fast', 1), ('hadoop', 1), ('is', 1), ('scalable', 1)]
Shuffled: {'hadoop': [1, 1], 'is': [1, 1], 'fast': [1], 'scalable': [1]}
Final Output: {'hadoop': 2, 'is': 2, 'fast': 1, 'scalable': 1}


In [3]:
%%writefile input.txt
hadoop is fast
hadoop is scalable

Writing input.txt


In [6]:
%%writefile wordcnt.py
from mrjob.job import MRJob

class MRWordCount(MRJob):
    def mapper(self, _, line):
        for word in line.split():
            yield word, 1

    def reducer(self, key, values):
        yield key, sum(values)

if __name__ == "__main__":
    MRWordCount.run()

Writing wordcnt.py


In [7]:
!python wordcnt.py input.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/wordcnt.root.20260507.073056.963545
Running step 1 of 1...
job output is in /tmp/wordcnt.root.20260507.073056.963545/output
Streaming final output from /tmp/wordcnt.root.20260507.073056.963545/output...
"fast"	1
"hadoop"	2
"scalable"	1
"is"	2
Removing temp directory /tmp/wordcnt.root.20260507.073056.963545...


Q2. Character Count - Count the frequency of each character (ignore spaces).
Input: big data

In [8]:
from collections import defaultdict

data = "big data"

# 🔸 Mapper
mapped = []
for char in data:
    if char != " ":   # ignore spaces
        mapped.append((char, 1))

print("Mapped:", mapped)

# 🔸 Shuffle & Sort
shuffled = defaultdict(list)
for key, value in mapped:
    shuffled[key].append(value)

print("Shuffled:", dict(shuffled))

# 🔸 Reducer
result = {}
for key, values in shuffled.items():
    result[key] = sum(values)

print("Final Output:", result)

Mapped: [('b', 1), ('i', 1), ('g', 1), ('d', 1), ('a', 1), ('t', 1), ('a', 1)]
Shuffled: {'b': [1], 'i': [1], 'g': [1], 'd': [1], 'a': [1, 1], 't': [1]}
Final Output: {'b': 1, 'i': 1, 'g': 1, 'd': 1, 'a': 2, 't': 1}


In [9]:
%%writefile input2.txt
big data

Writing input2.txt


In [10]:
%%writefile charactercount.py
from mrjob.job import MRJob

class MRCharCount(MRJob):

    def mapper(self, _, line):
        for char in line:
            if char != " ":
                yield char, 1

    def reducer(self, key, values):
        yield key, sum(values)

if __name__ == "__main__":
    MRCharCount.run()

Writing charactercount.py


In [11]:
!python charactercount.py input2.txt --runner=inline

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/charactercount.root.20260507.073138.478175
Running step 1 of 1...
job output is in /tmp/charactercount.root.20260507.073138.478175/output
Streaming final output from /tmp/charactercount.root.20260507.073138.478175/output...
"a"	2
"g"	1
"i"	1
"b"	1
"d"	1
"t"	1
Removing temp directory /tmp/charactercount.root.20260507.073138.478175...


Q3. Average Word Length (Per Word) - Compute the average length of each word.
Input: data science data big data

In [12]:
from collections import defaultdict

data = "data science data big data".split()

# 🔸 Mapper
mapped = []
for word in data:
    mapped.append((word, (len(word), 1)))

print("Mapped:", mapped)

# 🔸 Shuffle
shuffled = defaultdict(list)
for key, value in mapped:
    shuffled[key].append(value)

print("Shuffled:", dict(shuffled))

# 🔸 Reducer
result = {}
for key, values in shuffled.items():
    total_len = sum(v[0] for v in values)
    count = sum(v[1] for v in values)
    result[key] = total_len / count

print("Final Output:", result)

Mapped: [('data', (4, 1)), ('science', (7, 1)), ('data', (4, 1)), ('big', (3, 1)), ('data', (4, 1))]
Shuffled: {'data': [(4, 1), (4, 1), (4, 1)], 'science': [(7, 1)], 'big': [(3, 1)]}
Final Output: {'data': 4.0, 'science': 7.0, 'big': 3.0}


In [13]:
%%writefile input3.txt
data science data big data

Writing input3.txt


In [14]:
%%writefile averageword.py
from mrjob.job import MRJob

class MRAvgWordLength(MRJob):

    def mapper(self, _, line):
        for word in line.split():
            yield word, (len(word), 1)

    def reducer(self, key, values):
        total_len = 0
        count = 0
        for length, c in values:
            total_len += length
            count += c
        yield key, total_len / count

if __name__ == "__main__":
    MRAvgWordLength.run()

Writing averageword.py


In [15]:
!python averageword.py input3.txt --runner=inline

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/averageword.root.20260507.073209.727339
Running step 1 of 1...
job output is in /tmp/averageword.root.20260507.073209.727339/output
Streaming final output from /tmp/averageword.root.20260507.073209.727339/output...
"big"	3.0
"data"	4.0
"science"	7.0
Removing temp directory /tmp/averageword.root.20260507.073209.727339...


Q4. Global Average Word Length - Compute the average length of all words.
Input: hadoop mapreduce spark

In [16]:
data = "hadoop mapreduce spark".split()

# 🔸 Mapper
mapped = []
for word in data:
    mapped.append(("avg", (len(word), 1)))

print("Mapped:", mapped)

# 🔸 Shuffle
from collections import defaultdict
shuffled = defaultdict(list)

for key, value in mapped:
    shuffled[key].append(value)

print("Shuffled:", dict(shuffled))

# 🔸 Reducer
total_len = 0
count = 0

for values in shuffled["avg"]:
    total_len += values[0]
    count += values[1]

global_avg = total_len / count

print("Global Average:", global_avg)

Mapped: [('avg', (6, 1)), ('avg', (9, 1)), ('avg', (5, 1))]
Shuffled: {'avg': [(6, 1), (9, 1), (5, 1)]}
Global Average: 6.666666666666667


In [17]:
%%writefile input4.txt
hadoop mapreduce spark

Writing input4.txt


In [18]:
%%writefile globalavg.py

from mrjob.job import MRJob

class MRGlobalAvg(MRJob):

    def mapper(self, _, line):
        for word in line.split():
            yield "avg", (len(word), 1)

    def reducer(self, key, values):
        total_len = 0
        count = 0
        for length, c in values:
            total_len += length
            count += c
        yield key, total_len / count

if __name__ == "__main__":
    MRGlobalAvg.run()

Writing globalavg.py


In [19]:
!python globalavg.py input4.txt --runner=inline

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/globalavg.root.20260507.073235.332778
Running step 1 of 1...
job output is in /tmp/globalavg.root.20260507.073235.332778/output
Streaming final output from /tmp/globalavg.root.20260507.073235.332778/output...
"avg"	6.666666666666667
Removing temp directory /tmp/globalavg.root.20260507.073235.332778...


Q5. Perform Q1-Q4 on the file available on the link:
https://drive.google.com/file/d/16TIgKhcc2JH8jyJXfwouOV3y7ACX_aas/view?usp=sharing
• Also find Top 5 most frequent words

In [21]:
from collections import defaultdict, Counter

# Read file
with open("shakespeare.txt", "r", encoding="utf-8") as f:
    text = f.read().lower()

words = text.split()

# -------------------
# Q1: Word Count
# -------------------
word_count = Counter(words)
print("Q1 Word Count:\n", dict(word_count))

# -------------------
# Q2: Character Count (ignore spaces)
# -------------------
char_count = Counter([c for c in text if c != " " and c != "\n"])
print("\nQ2 Character Count:\n", dict(char_count))

# -------------------
# Q3: Average Word Length (per word)
# -------------------
word_len = defaultdict(list)

for w in words:
    word_len[w].append(len(w))

avg_word_len = {k: sum(v)/len(v) for k,v in word_len.items()}
print("\nQ3 Avg Word Length:\n", avg_word_len)

# -------------------
# Q4: Global Average Word Length
# -------------------
total_len = sum(len(w) for w in words)
count = len(words)

global_avg = total_len / count
print("\nQ4 Global Avg Word Length:", global_avg)

# -------------------
# Top 5 Words
# -------------------
top5 = word_count.most_common(5)
print("\nTop 5 Words:\n", top5)

Q1 Word Count:
 {'\ufeffthe': 1, 'project': 320, 'gutenberg': 250, 'ebook': 13, 'of': 18126, 'the': 27729, 'complete': 243, 'works': 268, 'william': 311, 'shakespeare,': 2, 'by': 4310, 'shakespeare': 270, 'this': 5930, 'is': 9168, 'for': 8000, 'use': 509, 'anyone': 5, 'anywhere': 4, 'at': 2459, 'no': 2864, 'cost': 35, 'and': 26099, 'with': 7981, 'almost': 158, 'restrictions': 2, 'whatsoever.': 3, 'you': 10696, 'may': 1762, 'copy': 21, 'it,': 529, 'give': 1288, 'it': 5879, 'away': 346, 'or': 3100, 're-use': 2, 'under': 292, 'terms': 67, 'license': 20, 'included': 3, 'online': 4, 'www.gutenberg.org': 2, '**': 4, 'a': 14436, 'copyrighted': 2, 'ebook,': 2, 'details': 1, 'below': 33, 'please': 338, 'follow': 235, 'copyright': 243, 'guidelines': 1, 'in': 10730, 'file.': 1, 'title:': 1, 'author:': 1, 'posting': 5, 'date:': 3, 'september': 1, '1,': 1, '2011': 1, '[ebook': 1, '#100]': 1, 'release': 9, 'january,': 1, '1994': 1, 'language:': 2, 'english': 125, '***': 8, 'start': 24, 'works--willi

In [22]:
%%writefile analysisQ5.py
from mrjob.job import MRJob

class MRFullAnalysis(MRJob):

    def mapper(self, _, line):
        words = line.lower().split()

        for word in words:
            # Q1: word count
            yield ("word", word), 1

            # Q3: word length
            yield ("length", word), (len(word), 1)

            # Q4: global avg
            yield ("global", "avg"), (len(word), 1)

        # Q2: character count
        for char in line:
            if char not in [" ", "\n"]:
                yield ("char", char), 1

    def reducer(self, key, values):
        category, item = key

        if category == "word":
            yield ("WordCount", item), sum(values)

        elif category == "char":
            yield ("CharCount", item), sum(values)

        elif category == "length":
            total_len = 0
            count = 0
            for v in values:
                total_len += v[0]
                count += v[1]
            yield ("AvgWordLength", item), total_len / count

        elif category == "global":
            total_len = 0
            count = 0
            for v in values:
                total_len += v[0]
                count += v[1]
            yield ("GlobalAvg", "all"), total_len / count

if __name__ == "__main__":
    MRFullAnalysis.run()

Writing analysisQ5.py


In [23]:
!python analysisQ5.py shakespeare.txt --runner=inline

Streaming output truncated to the last 5000 lines.
["WordCount", "ulcerous,"]	1
["WordCount", "ulysses!"]	1
["WordCount", "ulysses"]	16
["WordCount", "ulysses'"]	1
["WordCount", "ulysses,"]	13
["WordCount", "ulysses."]	82
["WordCount", "ulysses?"]	1
["WordCount", "um."]	1
["WordCount", "umber"]	1
["WordCount", "umber'd"]	1
["WordCount", "umbra"]	1
["WordCount", "umbrage,"]	1
["WordCount", "umfrevile"]	1
["WordCount", "umpire"]	3
["WordCount", "umpires"]	1
["WordCount", "un"]	4
["WordCount", "un-feeling;"]	1
["WordCount", "unable"]	4
["WordCount", "unable."]	1
["WordCount", "unaccommodated"]	1
["WordCount", "unaccompanied"]	1
["WordCount", "unaccustom'd"]	5
["WordCount", "unaching"]	1
["WordCount", "unacquainted"]	2
["WordCount", "unacquainted-"]	1
["WordCount", "unactive,"]	1
["WordCount", "unadvis'd"]	2
["WordCount", "unadvis'd,"]	2
["WordCount", "unadvised"]	2
["WordCount", "unadvisedly"]	1
["WordCount", "unagreeable"]	1
["WordCount", "unanel'd,"]	1
["WordCount", "unanswer'd."]	1
["W

In [24]:
from collections import Counter
# reuse words list
top5 = Counter(words).most_common(5)
print(top5)

[('the', 27729), ('and', 26099), ('i', 19540), ('to', 18762), ('of', 18126)]


Q6. Compute average marks for each student.
Input:
A 80
B 70
A 90
B 60
A 100

In [25]:
from collections import defaultdict

data = ["A 80", "B 70", "A 90", "B 60", "A 100"]

# 🔸 Mapper
mapped = []
for line in data:
    name, marks = line.split()
    mapped.append((name, (int(marks), 1)))

print("Mapped:", mapped)

# 🔸 Shuffle
shuffled = defaultdict(list)
for key, value in mapped:
    shuffled[key].append(value)

print("Shuffled:", dict(shuffled))

# 🔸 Reducer
result = {}
for key, values in shuffled.items():
    total = sum(v[0] for v in values)
    count = sum(v[1] for v in values)
    result[key] = total / count

print("Final Output:", result)

Mapped: [('A', (80, 1)), ('B', (70, 1)), ('A', (90, 1)), ('B', (60, 1)), ('A', (100, 1))]
Shuffled: {'A': [(80, 1), (90, 1), (100, 1)], 'B': [(70, 1), (60, 1)]}
Final Output: {'A': 90.0, 'B': 65.0}


In [26]:
%%writefile input6.txt
A 80
B 70
A 90
B 60
A 100

Writing input6.txt


In [27]:
%%writefile studentavg.py
from mrjob.job import MRJob

class MRStudentAvg(MRJob):

    def mapper(self, _, line):
        name, marks = line.split()
        yield name, (int(marks), 1)

    def reducer(self, key, values):
        total = 0
        count = 0
        for marks, c in values:
            total += marks
            count += c
        yield key, total / count

if __name__ == "__main__":
    MRStudentAvg.run()

Writing studentavg.py


In [28]:
!python studentavg.py input6.txt --runner=inline

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/studentavg.root.20260507.073743.674766
Running step 1 of 1...
job output is in /tmp/studentavg.root.20260507.073743.674766/output
Streaming final output from /tmp/studentavg.root.20260507.073743.674766/output...
"A"	90.0
"B"	65.0
Removing temp directory /tmp/studentavg.root.20260507.073743.674766...


Q7. Compute average salary per department and Highest Paid Department (Based on Average
Salary)
Input:
HR 50000
IT 70000
HR 60000
IT 80000

In [30]:
from collections import defaultdict

data = ["HR 50000", "IT 70000", "HR 60000", "IT 80000"]

# 🔸 Mapper
mapped = []
for line in data:
    dept, salary = line.split()
    mapped.append((dept, (int(salary), 1)))

print("Mapped:", mapped)

# 🔸 Shuffle
shuffled = defaultdict(list)
for key, value in mapped:
    shuffled[key].append(value)

print("Shuffled:", dict(shuffled))

# 🔸 Reducer (Average Salary)
avg_salary = {}
for key, values in shuffled.items():
    total = sum(v[0] for v in values)
    count = sum(v[1] for v in values)
    avg_salary[key] = total / count

print("Average Salary:", avg_salary)

# 🔸 Highest Paid Department
highest_dept = max(avg_salary, key=avg_salary.get)

print("Highest Paid Department:", highest_dept)

Mapped: [('HR', (50000, 1)), ('IT', (70000, 1)), ('HR', (60000, 1)), ('IT', (80000, 1))]
Shuffled: {'HR': [(50000, 1), (60000, 1)], 'IT': [(70000, 1), (80000, 1)]}
Average Salary: {'HR': 55000.0, 'IT': 75000.0}
Highest Paid Department: IT


In [31]:
%%writefile input7.txt
HR 50000
IT 70000
HR 60000
IT 80000

Writing input7.txt


In [32]:
%%writefile dept_salary.py
from mrjob.job import MRJob

class MRDeptSalary(MRJob):

    def mapper(self, _, line):
        dept, salary = line.split()
        yield dept, (int(salary), 1)

    def reducer(self, key, values):
        total = 0
        count = 0
        for salary, c in values:
            total += salary
            count += c
        yield key, total / count

if __name__ == "__main__":
    MRDeptSalary.run()

result = {"HR": 55000, "IT": 75000}
print(max(result, key=result.get))

Writing dept_salary.py


In [33]:
!python dept_salary.py input7.txt --runner=inline

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/dept_salary.root.20260507.073758.621510
Running step 1 of 1...
job output is in /tmp/dept_salary.root.20260507.073758.621510/output
Streaming final output from /tmp/dept_salary.root.20260507.073758.621510/output...
"HR"	55000.0
"IT"	75000.0
Removing temp directory /tmp/dept_salary.root.20260507.073758.621510...
IT


Q8. Computer average temperature per country
New York,38
London,29
Tokyo,35
New York,32
Delhi,45
Ambala,35

In [35]:
from collections import defaultdict

data = [
    "New York,38",
    "London,29",
    "Tokyo,35",
    "New York,32",
    "Delhi,45",
    "Ambala,35"
]

# 🔸 Mapper
mapped = []
for line in data:
    city, temp = line.split(",")
    mapped.append((city, (int(temp), 1)))

print("Mapped:", mapped)

# 🔸 Shuffle
shuffled = defaultdict(list)
for key, value in mapped:
    shuffled[key].append(value)

print("Shuffled:", dict(shuffled))

# 🔸 Reducer
result = {}
for key, values in shuffled.items():
    total = sum(v[0] for v in values)
    count = sum(v[1] for v in values)
    result[key] = total / count

print("Final Output:", result)

Mapped: [('New York', (38, 1)), ('London', (29, 1)), ('Tokyo', (35, 1)), ('New York', (32, 1)), ('Delhi', (45, 1)), ('Ambala', (35, 1))]
Shuffled: {'New York': [(38, 1), (32, 1)], 'London': [(29, 1)], 'Tokyo': [(35, 1)], 'Delhi': [(45, 1)], 'Ambala': [(35, 1)]}
Final Output: {'New York': 35.0, 'London': 29.0, 'Tokyo': 35.0, 'Delhi': 45.0, 'Ambala': 35.0}


In [36]:
%%writefile input8.txt
New York,38
London,29
Tokyo,35
New York,32
Delhi,45
Ambala,35

Writing input8.txt


In [37]:
%%writefile citytemp.py
from mrjob.job import MRJob

class MRTempAvg(MRJob):

    def mapper(self, _, line):
        city, temp = line.split(",")
        yield city, (int(temp), 1)

    def reducer(self, key, values):
        total = 0
        count = 0
        for temp, c in values:
            total += temp
            count += c
        yield key, total / count

if __name__ == "__main__":
    MRTempAvg.run()

Writing citytemp.py


In [38]:
!python citytemp.py input8.txt --runner=inline

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/citytemp.root.20260507.073817.400090
Running step 1 of 1...
job output is in /tmp/citytemp.root.20260507.073817.400090/output
Streaming final output from /tmp/citytemp.root.20260507.073817.400090/output...
"Ambala"	35.0
"Delhi"	45.0
"Tokyo"	35.0
"London"	29.0
"New York"	35.0
Removing temp directory /tmp/citytemp.root.20260507.073817.400090...


Q9. Redo Q8 with the dataset available on:
https://www.kaggle.com/datasets/heemalichaudhari/global-land-temperatures

In [40]:
import pandas as pd

# Load your CSV file
df = pd.read_csv("GlobalLandTemperatures_GlobalLandTemperaturesByMajorCity.csv")

# Remove missing values
df = df.dropna(subset=["AverageTemperature"])

# Compute average temperature per country
result = df.groupby("Country")["AverageTemperature"].mean()

# Show first few results
print(result.head())

Country
Afghanistan    14.342919
Angola         23.693046
Australia      15.190055
Bangladesh     25.490568
Brazil         22.847555
Name: AverageTemperature, dtype: float64


In [41]:
%%writefile tempcountryQ9.py
from mrjob.job import MRJob

class MRTempCountry(MRJob):

    def mapper(self, _, line):
        try:
            parts = line.split(",")

            # Skip header
            if parts[0] == "dt":
                return

            temp = float(parts[1])
            country = parts[3]

            yield country, (temp, 1)

        except:
            pass

    def reducer(self, key, values):
        total = 0
        count = 0
        for temp, c in values:
            total += temp
            count += c
        yield key, total / count

if __name__ == "__main__":
    MRTempCountry.run()

Writing tempcountryQ9.py


In [42]:
!python tempcountryQ9.py GlobalLandTemperatures_GlobalLandTemperaturesByMajorCity.csv --runner=inline

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/tempcountryQ9.root.20260507.074141.771843
Running step 1 of 1...
job output is in /tmp/tempcountryQ9.root.20260507.074141.771843/output
Streaming final output from /tmp/tempcountryQ9.root.20260507.074141.771843/output...
"Abidjan"	26.163737197524014
"Addis Abeba"	17.525072662298964
"Ahmadabad"	26.52985294117645
"Aleppo"	17.37058733360228
"Alexandria"	20.31261702925729
"Ankara"	10.39211714770797
"Baghdad"	22.61434568965519
"Bangalore"	24.855895933014338
"Bangkok"	27.164733303650937
"Belo Horizonte"	21.071396469465682
"Berlin"	8.916233733417545
"Bogot\u00e1"	20.00226461843412
"Bombay"	26.63145215311006
"Bras\u00edlia"	21.727594942748066
"Cairo"	21.221259213759268
"Calcutta"	26.042152053712506
"Cali"	21.79702720403025
"Cape Town"	16.079079551521584
"Casablanca"	17.184157858613602
"Changchun"	4.923797927461138
"Chengdu"	10.638042143838764
"Chicago"	10.070643744030578
"Ch